# SPARC Example 23: Cross-Survey Schema Navigation

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.0**

The corpus combines four surveys with different schemas.
This example demonstrates the correct way to access per-ring data
regardless of survey — a common source of bugs in cross-survey code.

Key pattern:
```python
points = g.get('data') or g.get('rotation_curve', [])
```

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19563417  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
import json
import numpy as np

with open('rotation_curve_corpus_v7.json') as f:
    corpus = json.load(f)

# Demonstrate schema differences
examples = {
    'SPARC':        'DDO161',
    'THINGS':       next(g['galaxy'] for g in corpus['galaxies']
                         if g['survey']=='THINGS' and g.get('data')),
    'LITTLE_THINGS':next(g['galaxy'] for g in corpus['galaxies']
                         if g['survey']=='LITTLE_THINGS'),
    'WALLABY':      next(g['galaxy'] for g in corpus['galaxies']
                         if g['survey']=='WALLABY' and g.get('rotation_curve')),
}

print(f"{'Survey':<15} {'Galaxy':<20} {'Data key':<20} {'R field':<12} {'V field'}")
print('-' * 80)
for survey, name in examples.items():
    g = next(g for g in corpus['galaxies'] if g['galaxy']==name)
    if g.get('data'):
        key = 'data'
        r_field = list(g['data'][0].keys())[0] if g['data'] else '?'
        v_field = 'Vobs' if 'Vobs' in g['data'][0] else 'Vrot'
    elif g.get('rotation_curve'):
        key = 'rotation_curve'
        r_field = 'rad_kpc'
        v_field = 'vrot_kms'
    else:
        key, r_field, v_field = 'none', '-', '-'
    print(f"{survey:<15} {name:<20} {key:<20} {r_field:<12} {v_field}")


In [ ]:
# Safe cross-survey extraction function
def get_rc(g):
    """Extract (R, V) from any survey in the EPS corpus."""
    if g.get('data'):
        d = g['data']
        R = [p.get('Rad') or p.get('Vrot') for p in d]  # fallback
        R = [p.get('Rad', 0) for p in d]
        V = [p.get('Vobs') or p.get('Vrot', 0) for p in d]
    elif g.get('rotation_curve'):
        rc = g['rotation_curve']
        R  = [p['rad_kpc']  for p in rc]
        V  = [p['vrot_kms'] for p in rc]
    else:
        return None, None
    return np.array(R), np.array(V)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']
for (survey, name), color in zip(examples.items(), colors):
    g = next(g for g in corpus['galaxies'] if g['galaxy']==name)
    R, V = get_rc(g)
    if R is not None:
        ax.plot(R, V, 'o-', color=color, linewidth=1.5, markersize=4,
                label=f'{survey}: {name}')
ax.set_xlabel('Radius (kpc)', fontsize=12)
ax.set_ylabel('Velocity (km/s)', fontsize=12)
ax.set_title('Cross-Survey Rotation Curves — Single Extraction Function\n'
             'EPS Research Corpus v7.0', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('ex23_cross_survey.png', dpi=150, bbox_inches='tight')
plt.show()